<a href="https://colab.research.google.com/github/RajeshworM/Yield_Modelling_Automation/blob/main/Dashbaord.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<font size ='6'> <font color ='white'>**Basmati Market Price (Paddy and Rice) Trend**

In [1]:

from google.colab import files
uploaded = files.upload()

import pandas as pd
import plotly.graph_objects as go

file_name = list(uploaded.keys())[0]
df = pd.read_excel(file_name)

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date')

x_end = pd.Timestamp("2026-12-31")


commodity_market_map = {
    'Basmati Rice': ['Delhi', 'General'],
    'Basmati Paddy': ['Amritsar', 'Kaithal']
}


market_colors = {
    'Delhi': '#264653',
    'General': '#2A9D8F',
    'Amritsar': '#E76F51',
    'Kaithal': '#6A4C93'
}


fig = go.Figure()
trace_meta = []

for commodity, markets in commodity_market_map.items():
    for market in markets:
        for dtype in ['Actual', 'Forecast']:

            dff = df[
                (df['Commodity_Type'] == commodity) &
                (df['Market'] == market) &
                (df['Data_Type'] == dtype)
            ]

            if dff.empty:
                continue

            fig.add_trace(
                go.Scatter(
                    x=dff['Date'],
                    y=dff['Price'],
                    mode='lines',
                    name=market,
                    legendgroup=market,
                    showlegend=(dtype == 'Actual'),
                    line=dict(
                        color=market_colors[market],
                        width=3,
                        dash='solid' if dtype == 'Actual' else 'dot'
                    )
                )
            )

            trace_meta.append({
                'commodity': commodity,
                'market': market
            })


# SLICERS

commodity_buttons = [
    dict(
        label=c,
        method='update',
        args=[
            {'visible': [m['commodity'] == c for m in trace_meta]},
            {'title.text': f"{c} Price Trends"}   # ✅ dynamic title
        ]
    )
    for c in commodity_market_map
]

market_buttons = [
    dict(
        label=m,
        method='update',
        args=[{'visible': [x['market'] == m for x in trace_meta]}]
    )
    for m in market_colors
]


# FORECAST PERIOD SHADE

fig.add_vrect(
    x0="2025-05-01",
    x1="2026-12-31",
    fillcolor="rgba(230,230,230,0.6)",
    layer="below",
    line_width=0,
    annotation_text="Forecast Period",
    annotation_position="top left",
    annotation_font=dict(size=13, color="#555")
)


# LAYOUT — FINAL & CLIENT READY

fig.update_layout(
    title=dict(
        text="Basmati Rice Price Trends",
        x=0.5,
        y=0.92,
        font=dict(size=24, family="Arial Black", color="#1f2933")
    ),

    xaxis=dict(
        title="Year – Month",
        range=[df['Date'].min(), x_end],
        tickmode="linear",
        dtick="M3",
        tickformat="%b %Y",
        tickangle=-45,
        showticklabels=True,
        tickfont=dict(size=11, color="#374151"),
        showgrid=True,
        gridcolor="rgba(200,200,200,0.3)",
        rangeselector=dict(
            buttons=[
                dict(count=5, label="5Y", step="year", stepmode="backward"),
                dict(count=10, label="10Y", step="year", stepmode="backward"),
                dict(step="all", label="All")
            ],
            x=0,
            y=1.18
        )
    ),

    yaxis=dict(
        title="Price (Rs./Quintal)",
        tickfont=dict(size=12, color="#374151"),
        showgrid=True,
        gridcolor="rgba(200,200,200,0.3)"
    ),

    updatemenus=[
        dict(buttons=commodity_buttons, x=0.34, y=1.35),
        dict(buttons=market_buttons, x=0.56, y=1.35)
    ],

    annotations=[
        dict(text="Commodity", x=0.34, y=1.42, xref="paper", yref="paper", showarrow=False),
        dict(text="Market", x=0.56, y=1.42, xref="paper", yref="paper", showarrow=False)
    ],

    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.25,
        font=dict(size=13)
    ),

    margin=dict(b=120),
    template="plotly_white",
    height=720
)


# EXPORT INTERACTIVE HTML

html_file = "Basmati_Price_Dashboard.html"
fig.write_html(html_file)

files.download(html_file)


# SHOW

fig.show()


Saving Fact_Price.xlsx to Fact_Price.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<font size ='6'> <font color ='green'>**Basmati Export Quantity and Price Trend**

In [ ]:

from google.colab import files
uploaded = files.upload()

# IMPORTS

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


# LOAD

file_name = list(uploaded.keys())[0]
df = pd.read_excel(file_name)

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date')


# AUTO-DETECT FORECAST PERIOD

forecast_df = df[df['Data_Type'] == 'Forecast']
forecast_start = forecast_df['Date'].min()
forecast_end = forecast_df['Date'].max()

x_end = df['Date'].max()

# COUNTRY LIST & COLOR MAP

countries = sorted(df['Country'].unique().tolist())

palette = px.colors.qualitative.Dark24
color_map = {c: palette[i % len(palette)] for i, c in enumerate(countries)}


# FUNCTION TO BUILD DASHBOARD

def build_export_figure(value_col, y_title, base_title):

    fig = go.Figure()
    trace_meta = []

    for country in countries:
        for dtype in ['Actual', 'Forecast']:

            dff = df[
                (df['Country'] == country) &
                (df['Data_Type'] == dtype)
            ]

            if dff.empty:
                continue

            fig.add_trace(
                go.Scatter(
                    x=dff['Date'],
                    y=dff[value_col],
                    mode='lines',
                    name=country,
                    legendgroup=country,
                    showlegend=(dtype == 'Actual'),
                    line=dict(
                        color=color_map[country],
                        width=3,
                        dash='solid' if dtype == 'Actual' else 'dot'
                    )
                )
            )

            trace_meta.append({'country': country})


    # COUNTRY SLICER

    country_buttons = [
        dict(
            label="All Countries",
            method='update',
            args=[
                {'visible': [True] * len(trace_meta)},
                {'title.text': base_title}
            ]
        )
    ]

    country_buttons += [
        dict(
            label=c,
            method='update',
            args=[
                {'visible': [m['country'] == c for m in trace_meta]},
                {'title.text': f"{base_title} – {c}"}
            ]
        )
        for c in countries
    ]


    # FORECAST BACKGROUND

    fig.add_vrect(
        x0=forecast_start,
        x1=forecast_end,
        fillcolor="rgba(99, 102, 241, 0.18)",
        layer="below",
        line_width=0
    )

    fig.add_annotation(
        x=forecast_start + (forecast_end - forecast_start) / 2,
        y=0.96,
        xref="x",
        yref="paper",
        text=f"Forecast Period<br>({forecast_start:%b %Y} – {forecast_end:%b %Y})",
        showarrow=False,
        font=dict(size=14, family="Arial Black", color="#1f2937"),
        bgcolor="rgba(255,255,255,0.65)",
        borderpad=6
    )


    # LAYOUT

    fig.update_layout(
        title=dict(
            text=base_title,
            x=0.5,
            y=0.92,
            font=dict(size=24, family="Arial Black")
        ),

        xaxis=dict(
            title="Year – Month",
            range=[df['Date'].min(), x_end],
            tickmode="linear",
            dtick="M3",
            tickformat="%b %Y",
            tickangle=-45,
            showgrid=True,
            gridcolor="rgba(200,200,200,0.3)",

            rangeselector=dict(
                buttons=[
                    dict(count=5, label="5 Years", step="year", stepmode="backward"),
                    dict(count=10, label="10 Years", step="year", stepmode="backward"),
                    dict(step="all", label="All")
                ],
                x=0.02,
                y=1.18
            )
        ),

        yaxis=dict(
            title=y_title,
            showgrid=True,
            gridcolor="rgba(200,200,200,0.3)"
        ),

        # SLICERS
        updatemenus=[
            dict(
                buttons=country_buttons,
                x=0.42,
                y=1.32,
                direction="down",
                showactive=True
            )
        ],

        #SLICER TITLES
        annotations=[
            dict(
                text="<b>Country</b>",
                x=0.42,
                y=1.40,
                xref="paper",
                yref="paper",
                showarrow=False,
                font=dict(size=14)
            ),
            dict(
                text="<b>Year Range</b>",
                x=0.02,
                y=1.28,
                xref="paper",
                yref="paper",
                showarrow=False,
                font=dict(size=13)
            )
        ],

        legend=dict(
            orientation="h",
            x=0.5,
            xanchor="center",
            y=-0.28,
            font=dict(size=13)
        ),

        margin=dict(b=130),
        template="plotly_white",
        height=740
    )

    return fig


# BUILD DASHBOARDS

fig_qty = build_export_figure(
    value_col="Export_Qty_MT",
    y_title="Export Quantity (MT)",
    base_title="Basmati Export Quantity Trend"
)

fig_price = build_export_figure(
    value_col="Export_Price",
    y_title="Export Price (Rs./Tonnes)",
    base_title="Basmati Export Price Trend"
)


# EXPORT HTML

fig_qty.write_html("Basmati_Export_Quantity_Dashboard.html")
fig_price.write_html("Basmati_Export_Price_Dashboard.html")

files.download("Basmati_Export_Quantity_Dashboard.html")
files.download("Basmati_Export_Price_Dashboard.html")


# SHOW

fig_qty.show()
fig_price.show()


Saving Fact_Export.xlsx to Fact_Export.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>